# Final clone-detection benchmark report

Rebuilds publication-ready figures from `../outputs/paper/kaggle/` without extracting or changing the supplied ZIP files. The report includes the overall benchmark tables, language breakdowns, and results for all four experiments: latent capacity, cross-dataset transfer, cross-language transfer, and feature ablation.

The builder validates the dataset inside every archive. Any inconsistent archive is excluded and recorded in `artifacts/report_input_warnings.txt`.

In [ ]:
# SPECTRA notebook progress: begin
from tqdm.auto import tqdm

def _spectra_enable_cell_progress():
    from IPython import get_ipython
    _shell = get_ipython()
    if _shell is None:
        return
    _old = getattr(_shell, "_spectra_cell_progress", None)
    if _old is not None:
        _old[0].close()
        for _event, _callback in _old[1:]:
            _shell.events.unregister(_event, _callback)
    _bar = tqdm(total=2, desc="Notebook cells", unit="cell", mininterval=2)
    def _after_cell(_result):
        if _bar.n < _bar.total:
            _bar.update(1)
        if getattr(_result, "error_in_exec", None) or getattr(_result, "error_before_exec", None):
            _bar.set_description("Notebook cell failed")
            _bar.close()
        elif _bar.n >= _bar.total:
            _bar.close()
    _shell.events.register("post_run_cell", _after_cell)
    _shell._spectra_cell_progress = (_bar, ("post_run_cell", _after_cell))

_spectra_enable_cell_progress()
# SPECTRA notebook progress: end

from pathlib import Path
import sys

HERE = Path.cwd().resolve()
PROJECT_ROOT = next((path for path in (HERE, *HERE.parents) if (path / 'research').is_dir() and (path.parent / 'outputs').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open this notebook from the repository tree so the outputs directory can be found.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.latent_graph_learning.reports.build_final_benchmark_report import main
REPORT = main()
ARTIFACTS = REPORT['artifacts']
print(f'Artifacts: {ARTIFACTS}')
print(f'Figures rebuilt: {len(REPORT["images"])}')

In [ ]:
from IPython.display import Image, Markdown, display

display(Markdown((ARTIFACTS / 'paper_report_summary.md').read_text(encoding='utf-8')))
for path in REPORT['images']:
    display(Markdown(f'### {path.stem.replace("_", " ").title()}'))
    display(Image(filename=str(path)))

if REPORT['warnings']:
    display(Markdown('### Input warnings'))
    display(Markdown('\n'.join(f'- {warning}' for warning in REPORT['warnings'])))